
# Data Storage and Statistics (Member 2)

This notebook focuses on the **Data Storage and Statistics** component of our website monitoring project. Its purpose is to take the raw monitoring results produced by Member 1 (a list of dictionaries called `results`), store them in a structured format, and compute useful summary statistics.  

The notebook performs the following tasks:

- **Convert the raw results into a table**: It uses `pandas.DataFrame` to transform a list of dictionaries into a tabular structure for easier manipulation.
- **Persist the data to disk**: It saves the raw monitoring dataset to a CSV file so that it can be reviewed or analysed later.
- **Calculate key statistics**: It computes overall metrics (total number of checks, number of successful and failed checks, failure rate, and response‐time statistics) as well as a breakdown of failures by error type and per‑website statistics.
- **Save per‑website summaries**: It creates a second CSV file containing aggregated statistics for each monitored website.

Run this notebook after executing Member 1’s `check_website()` function and building the `results` list so that `results` is available in the notebook’s environment.


In [1]:
%run group_project.ipynb

import pandas as pd
import json
from datetime import datetime

# Ensure that the 'results' variable (list of dictionaries) is defined before running the cells below.
# Example structure of an element in `results`:
# {
#     'website': 'https://example.com',
#     'timestamp': '2026-03-19 12:00:00',
#     'status': 'success',
#     'http_status': 200,
#     'response_time': 0.123,
#     'error_type': None
# }


{'url': 'https://www.google.com', 'timestamp': '2026-03-21 01:32:29', 'status': 'failed', 'status_code': None, 'response_time': None, 'error_type': 'timeout'}
{'url': 'https://github.com', 'timestamp': '2026-03-21 01:32:35', 'status': 'success', 'status_code': 200, 'response_time': 1.273, 'error_type': None}
{'url': 'https://www.wikipedia.org', 'timestamp': '2026-03-21 01:32:36', 'status': 'success', 'status_code': 200, 'response_time': 1.063, 'error_type': None}
{'url': 'https://stackoverflow.com', 'timestamp': '2026-03-21 01:32:37', 'status': 'success', 'status_code': 200, 'response_time': 6.75, 'error_type': None}
{'url': 'https://www.youtube.com', 'timestamp': '2026-03-21 01:32:44', 'status': 'failed', 'status_code': None, 'response_time': None, 'error_type': 'timeout'}
{'url': 'https://web.telegram.org', 'timestamp': '2026-03-21 01:32:50', 'status': 'success', 'status_code': 200, 'response_time': 1.054, 'error_type': None}
{'url': 'https://www.cloudflare.com', 'timestamp': '2026-0

In [2]:

# Convert raw monitoring results into a DataFrame
try:
    df = pd.DataFrame(results)
except NameError:
    raise NameError("The variable 'results' is not defined. Please run Member1's monitoring code to populate it.")

# Display the first few rows of the raw data
df.head()

# Save raw results to CSV
raw_csv_path = 'monitoring_results_raw.csv'
df.to_csv(raw_csv_path, index=False)
print(f"Raw monitoring data saved to {raw_csv_path}")


Raw monitoring data saved to monitoring_results_raw.csv


In [3]:

# Compute overall summary statistics

total_checks = len(df)
successful_checks = (df['status'] == 'success').sum()
failed_checks = total_checks - successful_checks
failure_rate = failed_checks / total_checks if total_checks > 0 else float('nan')

response_times = df['response_time'].dropna()
avg_response_time = response_times.mean() if not response_times.empty else float('nan')
min_response_time = response_times.min() if not response_times.empty else float('nan')
max_response_time = response_times.max() if not response_times.empty else float('nan')

print("Overall Summary Statistics:")
print(f"Total checks: {total_checks}")
print(f"Successful checks: {successful_checks}")
print(f"Failed checks: {failed_checks}")
print(f"Failure rate: {failure_rate:.2%}")
print(f"Average response time (s): {avg_response_time:.3f}")
print(f"Minimum response time (s): {min_response_time:.3f}")
print(f"Maximum response time (s): {max_response_time:.3f}")

# Calculate failure breakdown by error type
failure_breakdown = (
    df[df['status'] == 'failure']
      .groupby('error_type')
      .size()
      .reset_index(name='failure_count')
      .sort_values('failure_count', ascending=False)
)

print("Failure breakdown by error_type:")
print(failure_breakdown)

# Calculate per-website statistics
per_website_stats = (
    df.groupby('url')
      .agg(
          total_checks=('status', 'count'),
          failures=('status', lambda x: (x != 'success').sum()),
          failure_rate=('status', lambda x: ((x != 'success').sum() / len(x)) if len(x) else float('nan')),
          avg_response_time=('response_time', 'mean'),
          min_response_time=('response_time', 'min'),
          max_response_time=('response_time', 'max'),
      )
      .reset_index()
)

# Save per-website summary to CSV
summary_csv_path = 'monitoring_results_summary_by_site.csv'
per_website_stats.to_csv(summary_csv_path, index=False)
print(f"Per-website summary saved to {summary_csv_path}")

# Display per-website statistics
per_website_stats

Overall Summary Statistics:
Total checks: 61
Successful checks: 34
Failed checks: 27
Failure rate: 44.26%
Average response time (s): 2.727
Minimum response time (s): 0.995
Maximum response time (s): 10.510
Failure breakdown by error_type:
Empty DataFrame
Columns: [error_type, failure_count]
Index: []
Per-website summary saved to monitoring_results_summary_by_site.csv


,url,total_checks,failures,failure_rate,avg_response_time,min_response_time,max_response_time
0,https://alfabank.ru,1,1,1.0,NaN,NaN,NaN
1,https://github.com,1,0,0.0,1.273,1.273,1.273
2,https://lenta.ru,1,0,0.0,1.184,1.184,1.184
3,https://max.ru,1,1,1.0,NaN,NaN,NaN
4,https://ok.ru,1,1,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...
56,https://www.vtb.ru,1,0,0.0,1.424,1.424,1.424
57,https://www.wikipedia.org,1,0,0.0,1.063,1.063,1.063
58,https://www.wildberries.ru,1,1,1.0,NaN,NaN,NaN
59,https://www.yandex.ru,1,0,0.0,4.018,4.018,4.018
